State Management Deep Dive


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class BadListState(TypedDict):
    steps : list
    
def step_one(state : BadListState) -> dict:
    current = state["steps"]
    new_list = current + ["step_one"]
    print(f" step_one is returning : {new_list}")
    return {"steps" : new_list}

def step_two(state : BadListState) -> dict:
    current = state["steps"]
    new_list = current + ["step_two"]
    print(f" step_two is returning : {new_list}")
    return {"steps" : new_list}

builder  = StateGraph(BadListState)
builder.add_node("step_one",step_one)
builder.add_node("step_two",step_two)

builder.add_edge(START,"step_one")
builder.add_edge("step_one","step_two")
builder.add_edge("step_two",END)

graph = builder.compile()
result = graph.invoke({"steps": []})

print(result)

 step_one is returning : ['step_one']
 step_two is returning : ['step_one', 'step_two']
{'steps': ['step_one', 'step_two']}


Why does this pattern break?
In the sequential example above the list "worked" — but only because each node could see the previous node's output. The moment you have parallel branches (two nodes that both run at the same time from the same starting state), the overwrite rule causes a conflict:

                ┌──────────────┐
                │  Both nodes  │
                │  read steps=[]│
     ┌──────────┴──────────────┴──────────┐
     ▼                                    ▼
branch_a returns {"steps": ["a"]}   branch_b returns {"steps": ["b"]}
                         │
                  LangGraph merges:
                  whichever arrives second WINS
                  Final: steps = ["b"]   ← "a" is LOST!
The overwrite trap: with the default behaviour, the last writer wins and all earlier writes to the same key are silently discarded.

Reducers completely solve this.

Reducer 
A reducer is a function you attach to a state field that tells LangGraph how to combine the existing value with the new value a node returns.

Reducer signature
def my_reducer(old_value, new_value):
    # combine them however you like
    return combined_value
LangGraph calls this function automatically whenever a node returns that key. Instead of replacing old_value with new_value, LangGraph calls my_reducer(old_value, new_value) and stores the result.

How to attach a reducer
You use Python's Annotated type hint:

from typing import Annotated

class MyState(TypedDict):
    #         - the type    - the reducer function
    steps: Annotated[list, my_reducer]
That's it. The same field, the same list — just one extra annotation and LangGraph now calls your reducer instead of overwriting.

Without reducer	With reducer
state["steps"] = new_value	state["steps"] = reducer(old, new_value)
Last writer wins	You control the merge logic
Fragile with parallel nodes	Safe with parallel nodes

In [3]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from operator import add

class GoodListState(TypedDict):
    steps : Annotated[list,add]
    
def step_one(state : GoodListState) -> dict:
    print(f"step_one returns {state['steps']}")
    return {"steps" : ["step_one"]}

def step_two(state : GoodListState) -> dict:
    print(f"step_one returns {state['steps']}")
    return {"steps" : ["step_two"]}

builder  = StateGraph(GoodListState)
builder.add_node("step_one",step_one)
builder.add_node("step_two",step_two)

builder.add_edge(START,"step_one")
builder.add_edge("step_one","step_two")
builder.add_edge("step_two",END)

graph = builder.compile()
result = graph.invoke({"steps": []})

print(result)

step_one returns []
step_one returns ['step_one']
{'steps': ['step_one', 'step_two']}


In [11]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph, START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,AIMessage

class ChatState(TypedDict):
    messages : Annotated[list,add_messages]
    turn : int
    
def human_message(state : ChatState) -> dict:
    new_msg = HumanMessage(content = "What is the capital of France?")
    print(f"[human_input] appending : {new_msg.content}")
    print(f"Turn : {state['turn']}")
    return {
        "messages" : [new_msg],
        "turn" : state["turn"]+1
    }
    
def AI_message(state : ChatState) -> dict:
    new_msg = AIMessage(content = "The capital of France is Paris.")
    print(f"[ai_input] appending : {new_msg.content}")
    print(f"Turn : {state['turn']}")
    return {
        "messages" : [new_msg],
        "turn" : state["turn"]+1
    }
    
builder = StateGraph(ChatState)
builder.add_node("humanMessage",human_message)
builder.add_node("AiMessage",AI_message)

builder.add_edge(START,"humanMessage")
builder.add_edge("humanMessage","AiMessage")
builder.add_edge("AiMessage",END)

graph = builder.compile()
result = graph.invoke({"messages" : [],"turn": 0})
print(f"Turn: {result["turn"]}")
for msg in result["messages"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print (f" [{role}] : {msg.content}")

[human_input] appending : What is the capital of France?
Turn : 0
[ai_input] appending : The capital of France is Paris.
Turn : 1
Turn: 2
 [Human] : What is the capital of France?
 [AI] : The capital of France is Paris.


Simple Hardcoded chat application

In [15]:
from typing import TypedDict , Annotated
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,AIMessage

class ChattingState(TypedDict):
    messages : Annotated[list,add_messages]
    
def HumanMessage1(state : ChattingState)-> dict:
    new_msg = HumanMessage(content = "Where is my order")
    print(f"Convo 1 : {new_msg}")
    return {
        "messages" : [new_msg]
    }
        
def AIMessage1(state : ChattingState)-> dict:
    new_msg = AIMessage(content = "Can you provide your order ID?")
    print(f"Convo 1 : {new_msg}")
    return {
        "messages" : [new_msg]
    }
        
def HumanMessage2(state : ChattingState)-> dict:
    new_msg = HumanMessage(content = "12345")
    print(f"Convo 2 : {new_msg}")
    return {
        "messages" : [new_msg]
    }
        
def AIMessage2(state : ChattingState)-> dict:
    new_msg = AIMessage(content = "your order will be delivered tomorrow.")
    print(f"Convo 2 : {new_msg}")
    return {
        "messages" : [new_msg]
    }
    
    
builder = StateGraph(ChattingState)
builder.add_node("HumanMessage1",HumanMessage1)
builder.add_node("AIMessage1",AIMessage1)
builder.add_node("HumanMessage2",HumanMessage2)
builder.add_node("AIMessage2",AIMessage2)

builder.add_edge(START,"HumanMessage1")
builder.add_edge("HumanMessage1","AIMessage1")
builder.add_edge("AIMessage1","HumanMessage2")
builder.add_edge("HumanMessage2","AIMessage2")
builder.add_edge("AIMessage2",END)

graph = builder.compile()
result = graph.invoke({"messages":[]})

for msg in result["messages"]:
    role = "Human" if isinstance(msg , HumanMessage) else "AI"
    print(f"[{role}] : {msg.content}")

Convo 1 : content='Where is my order' additional_kwargs={} response_metadata={}
Convo 1 : content='Can you provide your order ID?' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
Convo 2 : content='12345' additional_kwargs={} response_metadata={}
Convo 2 : content='your order will be delivered tomorrow.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
[Human] : Where is my order
[AI] : Can you provide your order ID?
[Human] : 12345
[AI] : your order will be delivered tomorrow.


Program to differentiate operator.add() and add_messages()

In [20]:
# Using operator.add()

from typing import TypedDict , Annotated
from langgraph.graph import StateGraph,START,END
from operator import add
from langchain_core.messages import AIMessage

class ChattingState(TypedDict):
    messages : Annotated[list,add]

        
def AIMessage1(state : ChattingState)-> dict:
    print("Node 1 running.........")
    return {
            "messages" : [
                AIMessage(content = "Hel",id=1)
            ]
        }
        
def AIMessage2(state : ChattingState)-> dict:
    print("Node 2 running")
    return {
            "messages" : [
                AIMessage(content = "Hel",id=1)
            ]
        }
    
    
builder = StateGraph(ChattingState)
builder.add_node("AIMessage1",AIMessage1)
builder.add_node("AIMessage2",AIMessage2)

builder.add_edge(START,"AIMessage1")
builder.add_edge("AIMessage1","AIMessage2")
builder.add_edge("AIMessage2",END)

graph = builder.compile()
result = graph.invoke({"messages":[]})

for msg in result["messages"]:
    role = "AI"
    print(f"[{role}] : {msg.content}")


# Using add_messages
from typing import TypedDict , Annotated
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage

class ChattingState(TypedDict):
    messages : Annotated[list,add_messages]

        
def AIMessage1(state : ChattingState)-> dict:
    new_msg = AIMessage(content = "your order",id=1)
    print("Node 1 running.........")
    return {
        "messages" : [
            AIMessage(content = "Hel",id=1)
        ]
    }
        
def AIMessage2(state : ChattingState)-> dict:
    new_msg = AIMessage(content = "your order will be delivered tomorrow.",id=1)
    print("Node 2 running........")
    return {
            "messages" : [
                AIMessage(content = "Hello world",id=1)
            ]
        }
    
    
builder = StateGraph(ChattingState)
builder.add_node("AIMessage1",AIMessage1)
builder.add_node("AIMessage2",AIMessage2)

builder.add_edge(START,"AIMessage1")
builder.add_edge("AIMessage1","AIMessage2")
builder.add_edge("AIMessage2",END)

graph = builder.compile()
result = graph.invoke({"messages":[]})

for msg in result["messages"]:
    role = "AI"
    print(f"[{role}] : {msg.content}")
    
    
    

Node 1 running.........
Node 2 running
[AI] : Hel
[AI] : Hel
Node 1 running.........
Node 2 running........
[AI] : Hello world


Custom Reducer


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END

class CustState(TypedDict):
    score : Annotated[int,keep_max]
    
# Custom Reducer
def keep_max(old,new) :
    return max(old,new)

def student1(state : CustState):
    return {
        "score" : 64
    }
    
def student2(state : CustState):
    return {
        "score" : 181
    }
    
def student3(state : CustState):
    return {
        "score" : 11
    }
    
builder = StateGraph(CustState)
builder.add_node("student1",student1)
builder.add_node("student2",student2)
builder.add_node("student3",student3)

builder.add_edge(START,"student1")
builder.add_edge("student1","student2")
builder.add_edge("student2","student3")
builder.add_edge("student3",END)

graph = builder.compile()
result = graph.invoke({"score" : 0})

print(result)


{'score': 181}
